In [2]:
import re
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from transformers import pipeline

/Users/tori/Documents/prog/diploma/mvp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Разбиение текста

In [3]:
def split_text(text):
    parts = re.split(r"[.!?]|но|а|однако", text)
    return [p.strip() for p in parts if p.strip()]

## Инициализация моделей

In [14]:
# embeddings для BERTopic
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# BERTopic
topic_model = BERTopic(embedding_model=embedding_model, verbose=False, min_topic_size=2)

# sentiment модель (многоязычная)
sentiment_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8168.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8327.70it/s]


## Основная функция


In [12]:
corpus = [
    "Айфон получил новый дизайн и улучшенную камеру",
    "Новый смартфон работает быстрее предыдущей версии",
    "Батарея у телефона разряжается слишком быстро",
    "Экран стал ярче и приятнее для глаз",
    "Камера делает качественные фотографии даже ночью",
    
    "TikTok снова набирает популярность среди пользователей",
    "Алгоритмы Instagram плохо продвигают новые посты",
    "YouTube изменил рекомендации видео",
    "Социальные сети становятся менее интересными",
    "Новый тренд в TikTok быстро стал вирусным",
    
    "ChatGPT помогает писать тексты и код",
    "Искусственный интеллект активно развивается",
    "Нейросети начинают заменять людей в некоторых задачах",
    "AI используется для генерации изображений",
    "Многие компании внедряют искусственный интеллект",
    
    "Новая игра стала популярной среди стримеров",
    "Графика в игре выглядит устаревшей",
    "Игровая индустрия растет с каждым годом",
    "Игроки жалуются на баги в новой игре",
    "Геймплей стал более интересным и динамичным",
    
    "Этот мем стал популярным в интернете",
    "Пользователи активно делятся новыми мемами",
    "Тренд быстро набрал популярность и так же быстро исчез",
    "Интернет-культура меняется очень быстро",
    "Мемы становятся частью повседневной жизни",
    
    "Маркетинг в социальных сетях становится сложнее",
    "Реклама в интернете становится менее эффективной",
    "Бренды активно используют блогеров для продвижения",
    "Контент должен быть уникальным, чтобы привлечь внимание",
    "Алгоритмы платформ влияют на охваты",
    
    "Люди стали больше времени проводить в интернете",
    "Онлайн-образование становится популярнее",
    "Удаленная работа становится нормой",
    "Цифровые технологии меняют образ жизни",
    "Многие сервисы переходят в онлайн",
    
    "Новый ноутбук работает очень быстро и стабильно",
    "Производительность устройства заметно выросла",
    "Операционная система стала работать медленнее",
    "Обновление ухудшило производительность",
    "Приложения стали запускаться быстрее",
    
    "Видеоконтент становится популярнее текстового",
    "Короткие видео набирают больше просмотров",
    "Стриминг продолжает развиваться",
    "Платформы конкурируют за внимание пользователей",
    "Контент становится более персонализированным"
]

In [15]:
topic_model.fit(corpus)

In [16]:
def analyze_text(text):
    # Шаг 1: разбиваем
    sentences = split_text(text)

    # Шаг 2: темы через BERTopic
    topics, probs = topic_model.transform(sentences)

    # получаем слова тем
    topic_info = topic_model.get_topic_info()

    # маппинг topic_id → название
    topic_names = {}
    for topic_id in set(topics):
        if topic_id == -1:
            topic_names[topic_id] = "другое"
        else:
            words = topic_model.get_topic(topic_id)
            topic_names[topic_id] = words[0][0]  # главное слово темы

    # Шаг 3: sentiment + агрегация
    topic_scores = {}

    for sentence, topic_id in zip(sentences, topics):
        # sentiment
        result = sentiment_model(sentence)[0]
        label = result["label"]
        score = result["score"]

        # перевод в число
        if "1" in label or "2" in label:
            sent_value = -1
        elif "3" in label:
            sent_value = 0
        else:
            sent_value = 1

        weighted_score = sent_value * score

        topic_name = topic_names[topic_id]

        if topic_name not in topic_scores:
            topic_scores[topic_name] = []

        topic_scores[topic_name].append(weighted_score)

    # Шаг 4: финальная агрегация
    final = {}
    for topic, values in topic_scores.items():
        avg = np.mean(values)

        if avg > 0.2:
            final[topic] = "positive"
        elif avg < -0.2:
            final[topic] = "negative"
        else:
            final[topic] = "neutral"

    return final


## Пример

In [21]:
text = "Айфон красивый, цвет очень понравился, но камера слабая, зато работает быстро"
sentences_for_analysis = [
    "Телефон новый, батарея держит долго",
    "Камера делает качественные фото",
    "Дизайн смартфона очень красивый"
]
result = analyze_text(sentences_for_analysis)

print(result)

TypeError: expected string or bytes-like object, got 'list'

In [19]:
topics, probs = topic_model.transform(sentences_for_analysis)

In [23]:
print(topics)

[np.int64(-1), np.int64(-1), np.int64(-1)]
